In [1]:
## Initialize Hyperparameters and import libraries

import numpy as np
import torch

from model import *
from utilities import *
from loss_ftn import *

In [ ]:
PATH = "D:\\01_Datasets\\Curriculum Learning\\"#true_results\\result_Hy_"+f'{n:08d}'+'gamma'+str(0.0).zfill(2)+".npy"

data_dims = np.shape(np.load(PATH + "true_results_multiwl\\result_Hy_"+f'{0:08d}'+'k'+str(0.01).zfill(4)+".npy")[0])

In [3]:
n_train = 4500
n_test = 1000

batch_size = 250
epochs = 100

name = (n_train/1000)

layer_pretrain = 6
layer_finetune = 10

In [4]:
train_loader ,test_loader = data_loader(n_train, n_test, batch_size, wl_list, data_dims, 0, dz)

Loading Test Data: 100%|██████████| 1000/1000 [00:58<00:00, 17.07it/s]


In [ ]:
model_pretrain = FNOModel2d(modes=16, width=32, blocks=layer_pretrain).cuda()
model          = FNOModel2d(modes=16, width=32, blocks=layer_finetune).cuda()

In [6]:
lr_top = 0.005
step_size = 10
gamma = 0.7

In [7]:
name = (n_train/1000)
SAVE_pretrain = f"DIRTL_{name:.2f}k_pretrain_{layer_pretrain}FL.pth"
SAVE_model = f"DIRTL_{name * 2 :.2f}k_freeze_{layer_finetune}FL.pth"
SAVE_lc =  f"DIRTL_{name* 2 :.2f}k_freeze_{layer_finetune}FL_LC.npz"

In [ ]:
model_pretrain.load_state_dict(torch.load(SAVE_pretrain))
model = transfer(model_pretrain, model, layer_pretrain)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Train_rel_L1_arr = []
Train_rel_L2_arr = []
Test_rel_L1_arr = []
Test_rel_L2_arr = []

# Define StepLR scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=step_size,    # number of epochs before decay
    gamma=gamma             # decay factor, e.g., 0.1 reduces LR by 10x
)

loss = nn.MSELoss()

# gc.collect(k)
torch.cuda.empty_cache()

total_time = 0

for ep in range(epochs):
    t1 = default_timer()
    model.train()
    Train_mse = 0

    for input_shape, result in train_loader:
        input_shape, result = input_shape.cuda(), result.cuda()
        optimizer.zero_grad()
        
        out = model((input_shape))

        Train_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))

        Train_mse_temp.backward()
        
        optimizer.step()
        
        Train_mse += Train_mse_temp.detach() * batch_size

    scheduler.step()

    model.eval()
    Test_mse = 0.0
    with torch.no_grad():
        for input_shape, result in test_loader:
            input_shape, result = input_shape.cuda(), result.cuda()

            out = model((input_shape))
            Test_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))
            Test_mse += Test_mse_temp.detach() * batch_size

    Train_mse /= len(train_loader.dataset)
    Test_mse /= len(test_loader.dataset)

    Train_rmse = np.sqrt(Train_mse.item())
    Test_rmse = np.sqrt(Test_mse.item())

    if SAVE_lc:
        with torch.no_grad():
            model.eval()
            _, _, rel_L1, rel_L2, _ = rel_err(model, train_loader)
            Train_rel_L1_arr.append(np.mean(rel_L1))
            Train_rel_L2_arr.append(np.mean(rel_L2))

            _, _, rel_L1, rel_L2, _ = rel_err(model, test_loader)
            Test_rel_L1_arr.append(np.mean(rel_L1))
            Test_rel_L2_arr.append(np.mean(rel_L2))
        
    t2 = default_timer()
    total_time += t2 - t1

    print(f"Epoch {ep+1}, Time: {t2-t1:.2f}s, Train RMSE: {Train_rmse:.4f}, Test RMSE: {Test_rmse:.4f}")

print(f"total time: {total_time:.2f}")

if SAVE_model:
    torch.save(model.state_dict(), SAVE_model)

if SAVE_lc:
    np.savez(SAVE_lc,
        Train_rel_L1=Train_rel_L1_arr,
        Train_rel_L2=Train_rel_L2_arr,
        Test_rel_L1=Test_rel_L1_arr,
        Test_rel_L2=Test_rel_L2_arr)

Epoch 1, Time: 127.17s, Train RMSE: 0.2527, Test RMSE: 0.1306
Epoch 2, Time: 123.12s, Train RMSE: 0.1618, Test RMSE: 0.1195
Epoch 3, Time: 123.03s, Train RMSE: 0.1545, Test RMSE: 0.1049
Epoch 4, Time: 123.36s, Train RMSE: 0.1398, Test RMSE: 0.1207
Epoch 5, Time: 123.09s, Train RMSE: 0.1316, Test RMSE: 0.1004
Epoch 6, Time: 123.22s, Train RMSE: 0.1295, Test RMSE: 0.0980
Epoch 7, Time: 123.07s, Train RMSE: 0.1167, Test RMSE: 0.1016
Epoch 8, Time: 123.06s, Train RMSE: 0.0884, Test RMSE: 0.1496
Epoch 9, Time: 123.10s, Train RMSE: 0.0844, Test RMSE: 0.0887
Epoch 10, Time: 122.90s, Train RMSE: 0.0961, Test RMSE: 0.0875
Epoch 11, Time: 123.02s, Train RMSE: 0.1175, Test RMSE: 0.0911
Epoch 12, Time: 123.56s, Train RMSE: 0.0826, Test RMSE: 0.0889
Epoch 13, Time: 123.00s, Train RMSE: 0.1138, Test RMSE: 0.0973
Epoch 14, Time: 123.14s, Train RMSE: 0.0913, Test RMSE: 0.1187
Epoch 15, Time: 122.95s, Train RMSE: 0.0786, Test RMSE: 0.0883
Epoch 16, Time: 123.08s, Train RMSE: 0.0631, Test RMSE: 0.0805
E